### Model training

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
# set display columns to max so that we can view all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore")

# Optional: nicer display for pandas in notebooks
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

# Update this path if your CSV is in a different location within the project
csv_path = Path("data/clean/US_Accidents_March23_NY_Clean.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found at {csv_path}. Please place the file there or change the path in this cell.")

# Read CSV (use low_memory=False to avoid dtype warnings for large CSVs)
df = pd.read_csv(csv_path, low_memory=False)
print("Loaded:", csv_path, "-> shape:", df.shape)

# Show the first 5 rows so you can inspect column names and sample values
df.head()

Loaded: data\clean\US_Accidents_March23_NY_Clean.csv -> shape: (307740, 47)


,ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,Timezone,Weather_Timestamp,Temperature(F),Wind_Chill(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Weekday,Month,Day,Day_of_week,Is_weekend,Duration_min,Weather_raw,Weather_Short,Season,Severity_Class,Street_encoded
0,A-194257,3,2016-11-30 16:25:41,2016-11-30 17:10:30,41.139324,-74.169876,0.0100,Right lane blocked due to accident on I-87 Southbound around Exit 15A NY-17.,Route 17,Sloatsburg,Rockland,NY,US/Eastern,2016-11-30 16:53:00,53.1,51.0,1.0,29.81,93.0,Overcast,East,11.5,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,overcast,Cloudy/Clear,Autumn,Major,118
1,A-194258,3,2016-11-30 15:40:06,2016-11-30 17:09:40,41.118748,-74.159744,1.4925,Accident on I-87 Southbound between Exit 14B County Hwy-89 Airmont Rd and Exit 15 I-287.,I-287 S,Suffern,Rockland,NY,US/Eastern,2016-11-30 15:53:00,53.1,51.0,1.0,29.83,93.0,Overcast,ENE,8.1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.566667,overcast,Cloudy/Clear,Autumn,Major,141
2,A-194259,3,2016-11-30 15:26:18,2016-11-30 16:56:02,41.014034,-73.717239,1.1700,Accident on I-287 Cross Westchester Expy Eastbound between Exit 9A I-684 and Exit 9S-N I-684.,Hutchinson River Pkwy S,White Plains,Westchester,NY,US/Eastern,2016-11-30 15:56:00,52.0,51.0,2.0,29.81,93.0,Light Rain,East,17.3,0,0,0,0,1,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.733333,light rain,Rain/Precip,Autumn,Major,1682
3,A-194260,3,2016-11-30 16:14:11,2016-11-30 16:59:00,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Westbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,rain,Rain/Precip,Autumn,Major,1895
4,A-194261,3,2016-11-30 16:17:57,2016-11-30 17:02:44,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Eastbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.783333,rain,Rain/Precip,Autumn,Major,1895


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 47 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID                 307740 non-null  object 
 1   Severity           307740 non-null  int64  
 2   Start_Time         307740 non-null  object 
 3   End_Time           307740 non-null  object 
 4   Start_Lat          307740 non-null  float64
 5   Start_Lng          307740 non-null  float64
 6   Distance(mi)       307740 non-null  float64
 7   Description        307739 non-null  object 
 8   Street             307740 non-null  object 
 9   City               307740 non-null  object 
 10  County             307740 non-null  object 
 11  State              307740 non-null  object 
 12  Timezone           307740 non-null  object 
 13  Weather_Timestamp  307740 non-null  object 
 14  Temperature(F)     307740 non-null  float64
 15  Wind_Chill(F)      307740 non-null  float64
 16  Vi

In [3]:
target_col = "Severity_Class" 
infra_cols = [
    'Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway',
    'Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop'
] 

numeric_cols = ['Distance(mi)','Temperature(F)','Visibility(mi)',
                        'Humidity(%)','Pressure(in)','Wind_Speed(mph)','Hour','Duration_min']

category_cols = ["Street_encoded", "County","Weather_Condition","Wind_Direction","Weather_Short","Season"]

feature_cols = numeric_cols + category_cols + infra_cols
# Combined categorical columns (infrastructure flags are categorical / indicator features)
print("Categorical columns ", category_cols + infra_cols)
print("Numeric columns ", numeric_cols )

Categorical columns  ['Street_encoded', 'County', 'Weather_Condition', 'Wind_Direction', 'Weather_Short', 'Season', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']
Numeric columns  ['Distance(mi)', 'Temperature(F)', 'Visibility(mi)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)', 'Hour', 'Duration_min']


In [4]:
df.isna().sum()

ID                   0
Severity             0
Start_Time           0
End_Time             0
Start_Lat            0
Start_Lng            0
Distance(mi)         0
Description          1
Street               0
City                 0
County               0
State                0
Timezone             0
Weather_Timestamp    0
Temperature(F)       0
Wind_Chill(F)        0
Visibility(mi)       0
Pressure(in)         0
Humidity(%)          0
Weather_Condition    0
Wind_Direction       0
Wind_Speed(mph)      0
Amenity              0
Bump                 0
Crossing             0
Give_Way             0
Junction             0
No_Exit              0
Railway              0
Roundabout           0
Station              0
Stop                 0
Traffic_Calming      0
Traffic_Signal       0
Turning_Loop         0
Hour                 0
Weekday              0
Month                0
Day                  0
Day_of_week          0
Is_weekend           0
Duration_min         0
Weather_raw          0
Weather_Sho

In [5]:
df.head(5)

,ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,Timezone,Weather_Timestamp,Temperature(F),Wind_Chill(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Weekday,Month,Day,Day_of_week,Is_weekend,Duration_min,Weather_raw,Weather_Short,Season,Severity_Class,Street_encoded
0,A-194257,3,2016-11-30 16:25:41,2016-11-30 17:10:30,41.139324,-74.169876,0.0100,Right lane blocked due to accident on I-87 Southbound around Exit 15A NY-17.,Route 17,Sloatsburg,Rockland,NY,US/Eastern,2016-11-30 16:53:00,53.1,51.0,1.0,29.81,93.0,Overcast,East,11.5,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,overcast,Cloudy/Clear,Autumn,Major,118
1,A-194258,3,2016-11-30 15:40:06,2016-11-30 17:09:40,41.118748,-74.159744,1.4925,Accident on I-87 Southbound between Exit 14B County Hwy-89 Airmont Rd and Exit 15 I-287.,I-287 S,Suffern,Rockland,NY,US/Eastern,2016-11-30 15:53:00,53.1,51.0,1.0,29.83,93.0,Overcast,ENE,8.1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.566667,overcast,Cloudy/Clear,Autumn,Major,141
2,A-194259,3,2016-11-30 15:26:18,2016-11-30 16:56:02,41.014034,-73.717239,1.1700,Accident on I-287 Cross Westchester Expy Eastbound between Exit 9A I-684 and Exit 9S-N I-684.,Hutchinson River Pkwy S,White Plains,Westchester,NY,US/Eastern,2016-11-30 15:56:00,52.0,51.0,2.0,29.81,93.0,Light Rain,East,17.3,0,0,0,0,1,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.733333,light rain,Rain/Precip,Autumn,Major,1682
3,A-194260,3,2016-11-30 16:14:11,2016-11-30 16:59:00,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Westbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,rain,Rain/Precip,Autumn,Major,1895
4,A-194261,3,2016-11-30 16:17:57,2016-11-30 17:02:44,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Eastbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.783333,rain,Rain/Precip,Autumn,Major,1895


In [6]:
# Work on a copy to be safe
df_clean = df.copy()


# Infrastructure flags -> integer (0/1)
try: df['Amenity'] = df['Amenity'].astype('Int8')
except: print("Amenity cast failed")
try: df['Bump'] = df['Bump'].astype('Int8')
except: print("Bump cast failed")
try: df['Crossing'] = df['Crossing'].astype('Int8')
except: print("Crossing cast failed")
try: df['Give_Way'] = df['Give_Way'].astype('Int8')
except: print("Give_Way cast failed")
try: df['Junction'] = df['Junction'].astype('Int8')
except: print("Junction cast failed")
try: df['No_Exit'] = df['No_Exit'].astype('Int8')
except: print("No_Exit cast failed")
try: df['Railway'] = df['Railway'].astype('Int8')
except: print("Railway cast failed")
try: df['Roundabout'] = df['Roundabout'].astype('Int8')
except: print("Roundabout cast failed")
try: df['Station'] = df['Station'].astype('Int8')
except: print("Station cast failed")
try: df['Stop'] = df['Stop'].astype('Int8')
except: print("Stop cast failed")
try: df['Traffic_Calming'] = df['Traffic_Calming'].astype('Int8')
except: print("Traffic_Calming cast failed")
try: df['Traffic_Signal'] = df['Traffic_Signal'].astype('Int8')
except: print("Traffic_Signal cast failed")
try: df['Turning_Loop'] = df['Turning_Loop'].astype('Int8')
except: print("Turning_Loop cast failed")

# Numeric columns -> float
try: df['Distance(mi)'] = df['Distance(mi)'].astype('float64')
except: print("Distance(mi) cast failed")
try: df['Temperature(F)'] = df['Temperature(F)'].astype('float64')
except: print("Temperature(F) cast failed")
try: df['Visibility(mi)'] = df['Visibility(mi)'].astype('float64')
except: print("Visibility(mi) cast failed")
try: df['Humidity(%)'] = df['Humidity(%)'].astype('float64')
except: print("Humidity(%) cast failed")
try: df['Pressure(in)'] = df['Pressure(in)'].astype('float64')
except: print("Pressure(in) cast failed")
try: df['Wind_Speed(mph)'] = df['Wind_Speed(mph)'].astype('float64')
except: print("Wind_Speed(mph) cast failed")
try: df['Hour'] = df['Hour'].astype('Int64')
except: 
    try: df['Hour'] = df['Hour'].astype('float64')
    except: print("Hour cast failed")
try: df['Duration_min'] = df['Duration_min'].astype('float64')
except: print("Duration_min cast failed")

# Categorical columns -> category
try: df['Street_encoded'] = df['Street_encoded'].astype('category')
except: print("Street_encoded cast failed")
try: df['City'] = df['City'].astype('category')
except: print("City cast failed")
try: df['County'] = df['County'].astype('category')
except: print("County cast failed")
try: df['Weather_Condition'] = df['Weather_Condition'].astype('category')
except: print("Weather_Condition cast failed")
try: df['Wind_Direction'] = df['Wind_Direction'].astype('category')
except: print("Wind_Direction cast failed")
try: df['Weather_Short'] = df['Weather_Short'].astype('category')
except: print("Weather_Short cast failed")
try: df['Season'] = df['Season'].astype('category')
except: print("Season cast failed")

# Description -> string (text)
if 'Description' in df.columns:
    try: df['Description'] = df['Description'].astype('string')
    except: print("Description cast failed")
        
# Start_Time and End_Time -> datetime
# uses errors='coerce' to convert invalid formats to NaT while preserving the column
if 'Start_Time' in df.columns:
    try:
        df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
        print("Start_Time cast to datetime64[ns]")
    except Exception:
        print("Start_Time cast failed")
else:
    print("Start_Time: MISSING")

if 'End_Time' in df.columns:
    try:
        df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
        print("End_Time cast to datetime64[ns]")
    except Exception:
        print("End_Time cast failed")
else:
    print("End_Time: MISSING")

# Target -> category (replace target_col with your target variable name)
try:
    df[target_col] = df[target_col].astype('category')
except Exception:
    print(f"{target_col} cast to category failed")

Start_Time cast to datetime64[ns]
End_Time cast to datetime64[ns]


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 47 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   ID                 307740 non-null  object        
 1   Severity           307740 non-null  int64         
 2   Start_Time         307740 non-null  datetime64[ns]
 3   End_Time           307740 non-null  datetime64[ns]
 4   Start_Lat          307740 non-null  float64       
 5   Start_Lng          307740 non-null  float64       
 6   Distance(mi)       307740 non-null  float64       
 7   Description        307739 non-null  string        
 8   Street             307740 non-null  object        
 9   City               307740 non-null  category      
 10  County             307740 non-null  category      
 11  State              307740 non-null  object        
 12  Timezone           307740 non-null  object        
 13  Weather_Timestamp  307740 non-null  object  

In [8]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

In [9]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --------------------------------------------------------------------------
# 1) Train/test split
# - Stratify by target so class proportions are preserved in train/test
# - This prevents accidental optimistic evaluation when classes are imbalanced
# --------------------------------------------------------------------------
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)


In [10]:
# --------------------------------------------------------------------------
# 2) Preprocessing pipelines
# - Numeric: median imputation + standard scaling (robust and works for most models)
# - Categorical & infra flags: most-frequent imputation + one-hot encoding
# - Keep preprocessing inside a pipeline to avoid train-test leakage and to make CV safe
# --------------------------------------------------------------------------

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),   # fill numeric missing values
    ('scale', StandardScaler())                      # scale numeric features for linear models
])

# Treat infra flags as categorical/indicator features; OneHotEncoder will create 0/1 columns
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent'))
    ,   # fill missing categories
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))  # safe for unseen categories
])

cat_cols = category_cols + infra_cols
df[cat_cols] = df[cat_cols].astype("string")
preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, category_cols )
], remainder='drop')

In [11]:
df[cat_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Street_encoded     307740 non-null  string
 1   County             307740 non-null  string
 2   Weather_Condition  307740 non-null  string
 3   Wind_Direction     307740 non-null  string
 4   Weather_Short      307740 non-null  string
 5   Season             307740 non-null  string
 6   Amenity            307740 non-null  string
 7   Bump               307740 non-null  string
 8   Crossing           307740 non-null  string
 9   Give_Way           307740 non-null  string
 10  Junction           307740 non-null  string
 11  No_Exit            307740 non-null  string
 12  Railway            307740 non-null  string
 13  Roundabout         307740 non-null  string
 14  Station            307740 non-null  string
 15  Stop               307740 non-null  string
 16  Traffic_Calming    3

In [12]:
# --------------------------------------------------------------------------
# 3) Baseline models
# - LogisticRegression: interpretable baseline with class_weight="balanced"
# - DecisionTree: simple nonlinear baseline
# - RandomForest: ensemble baseline capturing interactions and non-linearities
# Each model is wrapped in a pipeline that includes the preprocessor
# --------------------------------------------------------------------------

lr_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', multi_class='multinomial', max_iter=1000, random_state=RANDOM_SEED))
])

dt_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED))
])

rf_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1))
])

In [13]:
# Fit baselines and print evaluation (weighted F1 is primary metric)
print("Fitting baseline models...")
for name, pipe in [('LogisticRegression', lr_pipeline), ('DecisionTree', dt_pipeline), ('RandomForest', rf_pipeline)]:
    pipe.fit(X_train, y_train)                       # train model
    preds = pipe.predict(X_test)                     # predict on test set
    print(f"\n=== {name} ===")
    print("Weighted F1:", f1_score(y_test, preds, average='weighted'))  # primary metric
    print(classification_report(y_test, preds))     # precision/recall per class;
    ; 

Fitting baseline models...


KeyboardInterrupt: 

In [ ]:
import xgboost as xgb

In [ ]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
])

In [ ]:
# -----------------------
# Option: train with SMOTE (example)
# - Use imblearn Pipeline so SMOTE is applied only to training folds
# -----------------------
use_smote = False  # set True to try oversampling
if use_smote:
    xgb_smote_pipeline = ImbPipeline([
        ('preproc', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_SEED)),
        ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
    ])

In [ ]:
# -----------------------
# Quick baseline fit for XGBoost (default params)
# -----------------------
# print("\nFitting XGBoost baseline (default params)...")

# xgb_pipeline.fit(X_train, y_train)
# xgb_preds = xgb_pipeline.predict(X_test)
# print("XGBoost baseline weighted F1:", f1_score(y_test, xgb_preds, average='weighted'))
# print(classification_report(y_test, xgb_preds))

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

print("\nFitting XGBoost baseline (default params)...")

# encode labels (fit only on training labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# fit and predict with encoded labels
xgb_pipeline.fit(X_train, y_train_enc)
xgb_preds_enc = xgb_pipeline.predict(X_test)

# decode predictions back to original string labels for readable reports
xgb_preds = le.inverse_transform(xgb_preds_enc)
y_test_labels = le.inverse_transform(y_test_enc)

print("XGBoost baseline weighted F1:", f1_score(y_test_labels, xgb_preds, average='weighted'))
print(classification_report(y_test_labels, xgb_preds))

In [ ]:
# Import label encoder for consistent numeric labels
from sklearn.preprocessing import LabelEncoder  # convert string classes to integers

# Create and fit encoder on the training target (fit once on full training set)
le = LabelEncoder()  # instantiate encoder
le.fit(y_train)  # learn mapping from string labels to integers, e.g., {'Major':0,'Minor':1,'Moderate':2}

# Transform targets used for training, tuning, and evaluation
y_train_enc = le.transform(y_train)  # encoded training labels
y_test_enc  = le.transform(y_test)   # encoded test labels (use same encoder)
# If you sample for tuning, encode the sampled labels too
if len(X_train) > sample_n:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=sample_n, stratify=y_train, random_state=RANDOM_SEED
    )  # sample by original string labels
    y_tune_enc = le.transform(y_tune)  # encode sampled tuning labels
else:
    X_tune, y_tune = X_train, y_train
    y_tune_enc = le.transform(y_tune)  # encode full training labels

# Use the encoded labels when fitting RandomizedSearchCV
rand_search.fit(X_tune, y_tune_enc)  # fit search with numeric labels

# After search, evaluate and predict using encoded labels; inverse-transform for readable labels if needed
best_xgb = rand_search.best_estimator_  # best pipeline/estimator found
y_pred_enc = best_xgb.predict(X_test)  # predictions are encoded integers
y_pred = le.inverse_transform(y_pred_enc)  # convert back to original string labels for reporting

# If you compute metrics with sklearn, use encoded y_test_enc and y_pred_enc
from sklearn.metrics import f1_score
print('Best CV weighted F1:', rand_search.best_score_)  # CV score reported by RandomizedSearchCV
print('Test weighted F1:', f1_score(y_test_enc, y_pred_enc, average='weighted'))  # test metric on encoded labels


In [ ]:
# -----------------------
# Hyperparameter tuning (RandomizedSearchCV)
# - StratifiedKFold preserves class proportions across folds
# - Optimize for weighted F1 (primary metric)
# - Keep search compact for speed; expand ranges for full tuning
# -----------------------
param_dist = {
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.03, 0.1],
    'clf__n_estimators': [100, 300, 600],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rand_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=5,                 # reduce for speed; increase for thorough search
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
    error_score="raise"   # <- raise exception so you get the real traceback
)

print("\nRunning RandomizedSearchCV for XGBoost...")
t0 = time.time()


# Import label encoder for consistent numeric labels
from sklearn.preprocessing import LabelEncoder  # convert string classes to integers

# Create and fit encoder on the training target (fit once on full training set)
le = LabelEncoder()  # instantiate encoder
le.fit(y_train)  # learn mapping from string labels to integers, e.g., {'Major':0,'Minor':1,'Moderate':2}

# Transform targets used for training, tuning, and evaluation
y_train_enc = le.transform(y_train)  # encoded training labels
y_test_enc  = le.transform(y_test)   # encoded test labels (use same encoder)
# If you sample for tuning, encode the sampled labels too
if len(X_train) > sample_n:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=sample_n, stratify=y_train, random_state=RANDOM_SEED
    )  # sample by original string labels
    y_tune_enc = le.transform(y_tune)  # encode sampled tuning labels
else:
    X_tune, y_tune = X_train, y_train
    y_tune_enc = le.transform(y_tune)  # encode full training labels


# Use the encoded labels when fitting RandomizedSearchCV
rand_search.fit(X_tune, y_tune_enc)  # fit search with numeric labels

# rand_search.fit(X_train, y_train)
t1 = time.time()
print(f"RandomizedSearchCV done in {t1 - t0:.1f}s")


# best_xgb = rand_search.best_estimator_

# After search, evaluate and predict using encoded labels; inverse-transform for readable labels if needed
best_xgb = rand_search.best_estimator_  # best pipeline/estimator found
y_pred_enc = best_xgb.predict(X_test)  # predictions are encoded integers
y_pred = le.inverse_transform(y_pred_enc)  # convert back to original string labels for reporting

# print("Best params:", rand_search.best_params_)
# print("Best CV weighted F1:", rand_search.best_score_)
# If you compute metrics with sklearn, use encoded y_test_enc and y_pred_enc
from sklearn.metrics import f1_score
print('Best CV weighted F1:', rand_search.best_score_)  # CV score reported by RandomizedSearchCV
print('Test weighted F1:', f1_score(y_test_enc, y_pred_enc, average='weighted'))  # test metric on encoded labels

In [ ]:
#  !pip install catboost

In [ ]:
# %pip install lightgbm

In [ ]:
#  pip install xgboost

In [ ]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
])

In [ ]:
from sklearn.exceptions import NotFittedError
try:
    # fit only the preprocessor on training data to locate transformer error early
    preprocessor.fit(X_train)
    print("preprocessor.fit succeeded")
except Exception as e:
    print("preprocessor.fit failed with:")
    print(e)
    # Inspect X_train.sample for likely culprits
    print("\nSample rows that might cause problem:")
    print(X_train.sample(10))

In [ ]:
# show dtypes for columns used by the pipeline
print("Numeric columns:", numeric_cols)
print("Category columns:", category_cols)
print("Infra columns:", infra_cols)

print("\nDataFrame dtypes (columns of interest):")
print(df[numeric_cols + category_cols + infra_cols].dtypes)

# find which numeric column contains non-numeric values (samples)
for col in numeric_cols:
    if col in df.columns:
        # try to coerce a small sample; any non-convertible value will become NaN
        sample = df[col].dropna().astype(str).head(200)
        non_numeric = sample[~sample.apply(lambda v: v.replace('.', '', 1).lstrip('-').isdigit())]
        if len(non_numeric)>0:
            print(f"\nColumn {col} has non-numeric examples (showing up to 10):")
            print(non_numeric.unique()[:10])